In [ ]:
!pip install sentence-transformers -q

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import re
from sentence_transformers import SentenceTransformer, util as st_util
import torch

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/Licenta/medical_dataset_final.csv'

df = pd.read_csv(CSV_PATH, sep=';', on_bad_lines='skip')
df = df[['filename', 'rezumat', 'keywords']].copy()
df['rezumat'] = df['rezumat'].fillna('')

Mounted at /content/drive


In [ ]:
stopwords_ro = [
    'si', 'în', 'la', 'de', 'a', 'au', 'cu', 'pe', 'că', 'ca', 'este', 'sunt',
    'ale', 'al', 'ai', 'sa', 'se', 'nu', 'sau', 'care', 'prin', 'din', 'pentru',
    'un', 'o', 'lui', 'ei', 'ele', 'cei', 'cel', 'cea', 'acest', 'această',
    'aceasta', 'acesta', 'acestor', 'acestui', 'după', 'între', 'mai', 'fi',
    'fost', 'avea', 'face', 'poate', 'dar', 'astfel', 'atât', 'când', 'cât',
    'dacă', 'deci', 'dupa', 'fara', 'iar', 'le', 'li', 'lor', 'ne', 'ni',
    'până', 'sub', 'tot', 'toate', 'toți', 'una', 'unde', 'unor', 'unui',
    'asupra', 'față', 'fata', 'fapt', 'fel', 'fiecare', 'general', 'lângă',
    'mult', 'mulți', 'nivel', 'nivelul', 'acestea', 'aceștia', 'aceleași',
    'același', 'ani', 'an', 'prezenta', 'prezentate', 'prezentarea',
    'studiu', 'studiul', 'studii', 'cazuri', 'cazul', 'caz', 'lot',
    'pacienți', 'pacientilor', 'pacient', 'pacientul', 'clinice', 'clinic',
    'clinica', 'clinicii', 'diagnosticul', 'diagnostic', 'diagnosticarea',
    'tratament', 'tratamentul', 'tratamentului', 'rezultate', 'rezultatul',
    'concluzii', 'concluzie', 'metoda', 'metodă', 'materiale', 'material',
    'introducere', 'scop', 'obiectiv', 'obiective', 'evaluare', 'analiza',
    'evaluarea', 'utilizate', 'utilizat', 'utilizarea', 'efectuate',
    'efectuat', 'efectuarea', 'prezent', 'prezentă', 'parte', 'tip', 'tipul',
    'grupe', 'grupa', 'grupul', 'vârstă', 'vârsta', 'sex', 'perioada',
    'perioadă', 'luni', 'luna', 'zile', 'spital', 'spitalul', 'internați',
    'internare', 'copil', 'copii', 'copilul', 'sugar', 'adult', 'bolnav',
    'bolnavii', 'bolnavi', 'număr', 'numărul', 'mod', 'dintre', 'astfel',
    'întrucât', 'această', 'înainte', 'întro'
]

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)   # elimină punctuație
    text = re.sub(r'\d+', ' ', text)        # elimină cifre
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['rezumat_proc'] = df['rezumat'].apply(preprocess)

In [ ]:
vectorizer = TfidfVectorizer(
    stop_words=stopwords_ro,
    ngram_range=(1, 2),   # unigrame + bigrame
    max_df=0.85,          # ignoră termeni din >85% documente
    min_df=3,             # minim 3 documente
    max_features=5000,
    sublinear_tf=True     # smoothing logaritmic
)

tfidf_matrix = vectorizer.fit_transform(df['rezumat_proc'])
feature_names = vectorizer.get_feature_names_out()

print(f"Vocabular: {len(feature_names)} termeni")
print(f"Matrice TF-IDF: {tfidf_matrix.shape}")

Vocabular: 5000 termeni
Matrice TF-IDF: (1114, 5000)


In [ ]:
def get_top_keywords(doc_idx, n=5):
    row = tfidf_matrix[doc_idx].toarray().flatten()
    top_idx = np.argsort(row)[::-1][:n]
    return [(feature_names[i], round(row[i], 4)) for i in top_idx if row[i] > 0]

TOP_N = 5  # modifică aici numărul de keywords dorit

results = []
for i, row in df.iterrows():
    kw = get_top_keywords(i, n=TOP_N)
    results.append({
        'filename': row['filename'],
        'keywords_extracted': '; '.join([k for k, s in kw]),
        'keywords_original': row['keywords'],
    })

results_df = pd.DataFrame(results)

In [ ]:
def overlap_score(extracted, original):
    if pd.isna(original) or pd.isna(extracted):
        return 0
    ext_set = {w.strip() for w in re.split(r'[;,]', extracted.lower())}
    orig_set = {w.strip() for w in re.split(r'[;,]', original.lower())}
    matches = sum(1 for e in ext_set for o in orig_set if e and o and (e in o or o in e))
    return min(matches, len(orig_set)) / max(len(orig_set), 1)

results_df['overlap'] = results_df.apply(
    lambda r: overlap_score(r['keywords_extracted'], r['keywords_original']), axis=1
)
print(f"\nOverlap mediu cu keywords originale : {results_df['overlap'].mean():.2%}")
print(f"Documente cu ≥1 keyword comun       : {(results_df['overlap'] > 0).sum()} / {len(results_df)}")


Overlap mediu cu keywords originale : 30.63%
Documente cu ≥1 keyword comun       : 702 / 1114


In [ ]:
print("Se încarcă modelul semantic...")
sem_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model încărcat.")

SIM_THRESHOLD = 0.6  # prag similaritate cosinus — ajustează dacă e nevoie

Se încarcă modelul semantic...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model încărcat.


In [16]:
def parse_keywords(text):
    if pd.isna(text) or str(text).strip() == '':
        return []
    return [w.strip().lower() for w in re.split(r'[;,]', str(text)) if w.strip()]

def compute_metrics(extracted_str, original_str):
    extracted = parse_keywords(extracted_str)
    original  = parse_keywords(original_str)

    if not extracted or not original:
        return 0.0, 0.0, 0.0

    all_terms = extracted + original
    embeddings = sem_model.encode(all_terms, convert_to_tensor=True)
    emb_ext  = embeddings[:len(extracted)]
    emb_orig = embeddings[len(extracted):]

    matched_orig = set()  # fiecare keyword original se poate potrivi o singură dată
    tp = 0
    for i, e in enumerate(extracted):
        for j, o in enumerate(original):
            if j in matched_orig:
                continue
            string_match   = e in o or o in e
            cos_score      = st_util.cos_sim(emb_ext[i], emb_orig[j]).item()
            semantic_match = cos_score >= SIM_THRESHOLD
            if string_match or semantic_match:
                tp += 1
                matched_orig.add(j)
                break

    precision = tp / len(extracted)
    recall    = tp / len(original)
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

print("Calculez metrici semantice (poate dura 1-2 minute)...")
results_df[['precision', 'recall', 'f1']] = results_df.apply(
    lambda r: pd.Series(compute_metrics(r['keywords_extracted'], r['keywords_original'])),
    axis=1
)

avg_p  = results_df['precision'].mean()
avg_r  = results_df['recall'].mean()
avg_f1 = results_df['f1'].mean()

print(f"\n{'='*45}")
print(f"  Precision medie : {avg_p:.4f}  ({avg_p:.2%})")
print(f"  Recall mediu    : {avg_r:.4f}  ({avg_r:.2%})")
print(f"  F1 mediu        : {avg_f1:.4f}  ({avg_f1:.2%})")
print(f"{'='*45}")
print(f"  Documente cu F1 > 0 : {(results_df['f1'] > 0).sum()} / {len(results_df)}")


Calculez metrici semantice (poate dura 1-2 minute)...

  Precision medie : 0.3178  (31.78%)
  Recall mediu    : 0.4470  (44.70%)
  F1 mediu        : 0.3627  (36.27%)
  Documente cu F1 > 0 : 996 / 1114


In [ ]:
print("\n--- Primele 5 documente ---")
for _, r in results_df.head(5).iterrows():
    print(f"\nFișier  : {r['filename']}")
    print(f"  TF-IDF   : {r['keywords_extracted']}")
    print(f"  Original : {r['keywords_original']}")


--- Primele 5 documente ---

Fișier  : RMJ_2017_3_Art-08.pdf
  TF-IDF   : polipilor; formațiuni; variat; cercetarea; raport
  Original : tumori; maligne; studiu; retrospectiv; endoscopic

Fișier  : RMR_Nr-3_2011_Art-6.pdf
  TF-IDF   : testul; testele; off; dureros; investigațiile paraclinice
  Original : sindrom impingement; umăr; teste clinice; regiune deltoidiana

Fișier  : RJP_2017_2_RO_Art-05.pdf
  TF-IDF   : primitivă; diagnosticare; neuroimagistice; ventriculului; precisă
  Original : meduloblastom; copil

Fișier  : RJP_2012_1_Art-09.pdf
  TF-IDF   : gh; substitutiv; persistentă; hipoglicemie; hormon creștere
  Original : hipoglicemie persistentă; deﬁcit gh; tratament substitutiv

Fișier  : RJP_2015_4_RO_Art-07.pdf
  TF-IDF   : etice; transplantul; ultimele decade; screeningul; îngrijirilor
  Original : fibroză chistică; etică; screening; terapie genică; transplant; fertilitate


In [17]:
from IPython.display import display, HTML

N_PREVIEW = 20  # câte articole vrei să vezi

def make_html_table(df_preview):
    rows = ''
    for _, r in df_preview.iterrows():
        color = '#c8f7c5' if r['f1'] >= 0.5 else '#fde8d8' if r['f1'] > 0 else '#f5f5f5'
        rows += f"""
        <tr>
            <td style='font-size:12px;color:#555;'>{r['filename']}</td>
            <td style='color:#1a73e8;font-weight:500;'>{r['keywords_extracted']}</td>
            <td style='color:#555;'>{r['keywords_original']}</td>
            <td style='text-align:center;'>{r['precision']:.2f}</td>
            <td style='text-align:center;'>{r['recall']:.2f}</td>
            <td style='text-align:center;background:{color};font-weight:bold;'>{r['f1']:.2f}</td>
        </tr>"""

    html = f"""
    <h3 style='font-family:sans-serif;'>🔍 TF-IDF — Primele {N_PREVIEW} articole</h3>
    <table style='border-collapse:collapse;width:100%;font-family:sans-serif;font-size:13px;'>
        <thead>
            <tr style='background:#1a73e8;color:white;'>
                <th style='padding:8px;text-align:left;'>Fișier</th>
                <th style='padding:8px;text-align:left;'>Keywords extrase (TF-IDF)</th>
                <th style='padding:8px;text-align:left;'>Keywords originale</th>
                <th style='padding:8px;text-align:center;'>Precision</th>
                <th style='padding:8px;text-align:center;'>Recall</th>
                <th style='padding:8px;text-align:center;'>F1</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    <p style='font-family:sans-serif;font-size:12px;color:#888;margin-top:8px;'>
        🟢 F1 ≥ 0.5 &nbsp;|&nbsp; 🟠 F1 > 0 &nbsp;|&nbsp; ⚪ F1 = 0
    </p>
    """
    return html

display(HTML(make_html_table(results_df.head(N_PREVIEW))))



Fișier,Keywords extrase (TF-IDF),Keywords originale,Precision,Recall,F1
RMJ_2017_3_Art-08.pdf,polipilor; formațiuni; variat; cercetarea; raport,tumori; maligne; studiu; retrospectiv; endoscopic,0.40,0.40,0.40
RMR_Nr-3_2011_Art-6.pdf,testul; testele; off; dureros; investigațiile paraclinice,sindrom impingement; umăr; teste clinice; regiune deltoidiana,0.40,0.50,0.44
RJP_2017_2_RO_Art-05.pdf,primitivă; diagnosticare; neuroimagistice; ventriculului; precisă,meduloblastom; copil,0.00,0.00,0.00
RJP_2012_1_Art-09.pdf,gh; substitutiv; persistentă; hipoglicemie; hormon creștere,hipoglicemie persistentă; deﬁcit gh; tratament substitutiv,0.60,1.00,0.75
RJP_2015_4_RO_Art-07.pdf,etice; transplantul; ultimele decade; screeningul; îngrijirilor,fibroză chistică; etică; screening; terapie genică; transplant; fertilitate,0.60,0.50,0.55
RJS_2014_3_Art-09.pdf,capului și; zece; peste jumătate; capului; estetice,hemangiom; cavitate orală; perioral; propranolol; corticosteroizi; laser; intralezional; scleroterapie,0.40,0.25,0.31
RJP_2012_2_Art-13.pdf,gigantă; hidronefroză; hta; abdominală; secundară,hidronefroză gigantă; hipertensiune; copil,0.20,0.33,0.25
RJP_2012_2_Art-07.pdf,wheezing; recurent; familială; manifestările alergice; sexul masculin,wheezing recurent; copil; factori de risc; atopie,0.60,0.75,0.67
RJP_2011_1_Art-01.pdf,alimentară; intestinale; testul; teste; determinarea,alergie alimentară; diagnostic; copil,0.20,0.33,0.25
RMJ_2019_1_Art-11.pdf,th; dz și; dz; aveau; decât pacienții,diabet zaharat tip 2; tiroidită; autoimun; corelație,0.40,0.50,0.44
